In [ ]:
import os

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import deephit_cancer_comparison.constants as const
from deephit_cancer_comparison.constants import GRAPH_PATH, RESULTS_PATH

EVAL_TIMES = [const.TIMESTEP * i for i in range(1, const.T_MAX // const.TIMESTEP + 1)]

cancer_types = [
    "breast",
    "corpus",
    "kidney_parenchyma",
    "lung_and_bronchus",
    "melanoma_of_the_skin",
    "pancreas",
    "prostate",
    "thyroid",
    "urinary_bladder",
    "colon_and_rectum",
]

MAX_TIME_HORIZON = 120  # 10 years

for cancer_type in cancer_types:
    if not os.path.exists(GRAPH_PATH / cancer_type):
        os.makedirs(GRAPH_PATH / cancer_type, exist_ok=True)

In [ ]:
cancer_col_name_map = {
    "breast": "Breast",
    "corpus": "Corpus",
    "kidney_parenchyma": "Kidney Parenchyma",
    "melanoma_of_the_skin": "Melanoma",
    "lung_and_bronchus": "Lung & Bronchus",
    "pancreas": "Pancreas",
    "prostate": "Prostate",
    "thyroid": "Thyroid",
    "urinary_bladder": "Urinary Bladder",
    "colon_and_rectum": "Colorectal",
}

In [ ]:
for cancer_type in cancer_types:
    c_index_mean_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_CINDEX_FINAL_MEAN.csv")
    c_index_mean_df = c_index_mean_df.iloc[:, : len(EVAL_TIMES)]

    c_index_rot_df = c_index_mean_df.T.iloc[1:, :]
    c_index_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, c_index_rot_df.shape[0] + 1)]
    )
    c_index_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    c_index_std_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_CINDEX_FINAL_STD.csv")
    c_index_std_rot_df = c_index_std_df.T.iloc[1:, :]
    c_index_std_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, c_index_std_rot_df.shape[0] + 1)]
    )
    c_index_std_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    c_index_time = c_index_rot_df["time"].tolist()

    c_index_event_mean = c_index_rot_df["primary_event"].tolist()
    c_index_sae_mean = c_index_rot_df["sae_event"].tolist()

    c_index_event_std = c_index_std_rot_df["primary_event"].tolist()
    c_index_sae_std = c_index_std_rot_df["sae_event"].tolist()

    value_05 = [0.5] * len(c_index_time)

    plt.figure(figsize=(14, 9))

    plt.plot(
        c_index_time,
        c_index_event_mean,
        label="C-Index (Primary)",
        linestyle="--",
        linewidth=3,
        c="#023E8A",
    )
    plt.fill_between(
        c_index_time,
        [m - s for m, s in zip(c_index_event_mean, c_index_event_std)],
        [m + s for m, s in zip(c_index_event_mean, c_index_event_std)],
        color="#023E8A",
        alpha=0.1,
    )

    plt.plot(
        c_index_time,
        c_index_sae_mean,
        label="C-Index (OCM)",
        linestyle="--",
        linewidth=3,
        c="#00b4d8",
    )
    plt.fill_between(
        c_index_time,
        [m - s for m, s in zip(c_index_sae_mean, c_index_sae_std)],
        [m + s for m, s in zip(c_index_sae_mean, c_index_sae_std)],
        color="#00b4d8",
        alpha=0.1,
    )

    plt.plot(
        c_index_time, value_05, linestyle="--", c="gray", linewidth=3, label="Chance level (0.5)"
    )

    plt.xlabel("Time (Months)", fontsize=18, weight="bold")
    plt.ylabel("C-index", fontsize=18, weight="bold")
    plt.title(
        f"C-index Over Time for Primary + OCM Outcomes (Cancer Type: {cancer_col_name_map[cancer_type]})",
        fontsize=20,
        fontweight="bold",
        pad=20,
    )
    plt.legend(fontsize=16, loc="lower right", title="Events", title_fontsize=16)
    plt.ylim(0, 1)
    plt.xlim(c_index_time[0], MAX_TIME_HORIZON)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig(GRAPH_PATH / cancer_type / f"result_cindex_{const.TIMESTEP}.png")
    plt.show();

In [ ]:
for cancer_type in cancer_types:
    brier_mean_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_BRIER_FINAL_MEAN.csv")
    brier_mean_df = brier_mean_df.iloc[:, : len(EVAL_TIMES)]
    brier_rot_df = brier_mean_df.T.iloc[1:, :]
    brier_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, brier_rot_df.shape[0] + 1)]
    )
    brier_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    brier_std_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_BRIER_FINAL_STD.csv")
    brier_std_rot_df = brier_std_df.T.iloc[1:, :]
    brier_std_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, brier_std_rot_df.shape[0] + 1)]
    )
    brier_std_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)
    brier_time = brier_rot_df["time"].tolist()

    brier_event_mean = brier_rot_df["primary_event"].tolist()
    brier_sae_mean = brier_rot_df["sae_event"].tolist()

    brier_event_std = brier_std_rot_df["primary_event"].tolist()
    brier_sae_std = brier_std_rot_df["sae_event"].tolist()

    plt.figure(figsize=(14, 9))

    plt.plot(
        brier_time,
        brier_event_mean,
        label="Brier Score (Primary)",
        linestyle="--",
        linewidth=3,
        c="#023E8A",
    )
    plt.fill_between(
        brier_time,
        [m - s for m, s in zip(brier_event_mean, brier_event_std)],
        [m + s for m, s in zip(brier_event_mean, brier_event_std)],
        color="#023E8A",
        alpha=0.1,
    )

    plt.plot(
        brier_time,
        brier_sae_mean,
        label="Brier Score (OCM)",
        linestyle="--",
        linewidth=3,
        c="#00b4d8",
    )
    plt.fill_between(
        brier_time,
        [m - s for m, s in zip(brier_sae_mean, brier_sae_std)],
        [m + s for m, s in zip(brier_sae_mean, brier_sae_std)],
        color="#00b4d8",
        alpha=0.1,
    )

    value_025 = [0.25] * len(brier_time)

    plt.plot(
        brier_time, value_025, linestyle="--", c="gray", linewidth=3, label="Chance level (0.25)"
    )

    plt.xlabel("Time (Months)", fontsize=18, weight="bold")
    plt.ylabel("Brier Score", fontsize=18, weight="bold")
    plt.title(
        f"Brier Score Over Time for Primary + OCM Outcomes (Cancer Type: {cancer_col_name_map[cancer_type]})",
        fontsize=20,
        fontweight="bold",
        pad=20,
    )
    plt.legend(fontsize=16, loc="upper left", title="Events", title_fontsize=16)
    plt.ylim(0, 0.4)
    plt.xlim(c_index_time[0], MAX_TIME_HORIZON)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.savefig(GRAPH_PATH / cancer_type / f"result_brier_{const.TIMESTEP}.png")
    plt.show();

In [ ]:
performance_df = pd.read_csv(RESULTS_PATH / "performance.csv")

data = performance_df.to_dict(orient="list")

cancers = data["cancer"]
primary_ci = data["primary_cindex_mean"]
primary_ci_e = data["primary_cindex_std"]
sae_ci = data["sae_cindex_mean"]
sae_ci_e = data["sae_cindex_std"]

primary_bs = data["primary_brier_mean"]
primary_bs_e = data["primary_brier_std"]
sae_bs = data["sae_brier_mean"]
sae_bs_e = data["sae_brier_std"]

x = np.arange(len(cancers))
w = 0.35
color_primary = "#023E8A"
color_sae = "#6b8cda"

fig1, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(
    x - w / 2,
    primary_ci,
    w,
    yerr=primary_ci_e,
    label="Primary",
    color=color_primary,
    capsize=4,
    error_kw=dict(elinewidth=1.2),
)
ax1.bar(
    x + w / 2,
    sae_ci,
    w,
    yerr=sae_ci_e,
    label="OCM",
    color=color_sae,
    capsize=4,
    error_kw=dict(elinewidth=1.2),
)
ax1.axhline(0.5, linestyle="--", color="gray", linewidth=1.5, label="Chance level (0.5)")
ax1.set_xticks(x)
ax1.set_xticklabels(cancers, rotation=30, ha="right", fontsize=10)
ax1.set_ylabel("C-index", fontsize=12)
ax1.set_xlabel("Cancer Type", fontsize=12)
ax1.set_title("C-index by Cancer Type", fontsize=15, pad=15)
ax1.set_ylim(0.4, 1.05)
ax1.legend(fontsize=10)
ax1.yaxis.grid(True, linestyle=":", alpha=0.6)
ax1.set_axisbelow(True)

fig1.tight_layout()
fig1.savefig(RESULTS_PATH / "cancer_cindex.png", dpi=600)
plt.show()

# --- Brier Score ---
fig2, ax2 = plt.subplots(figsize=(14, 6))

ax2.bar(
    x - w / 2,
    primary_bs,
    w,
    yerr=primary_bs_e,
    label="Primary",
    color=color_primary,
    capsize=4,
    error_kw=dict(elinewidth=1.2),
)
ax2.bar(
    x + w / 2,
    sae_bs,
    w,
    yerr=sae_bs_e,
    label="OCM",
    color=color_sae,
    capsize=4,
    error_kw=dict(elinewidth=1.2),
)
ax2.axhline(0.25, linestyle="--", color="gray", linewidth=1.5, label="Chance level (0.25)")
ax2.set_xticks(x)
ax2.set_xticklabels(cancers, rotation=30, ha="right", fontsize=10)
ax2.set_ylabel("Brier Score", fontsize=12)
ax2.set_xlabel("Cancer Type", fontsize=12)
ax2.set_title("Brier Score by Cancer Type", fontsize=15, pad=15)
ax2.set_ylim(0, 0.28)
ax2.legend(fontsize=10)
ax2.yaxis.grid(True, linestyle=":", alpha=0.6)
ax2.set_axisbelow(True)

fig2.tight_layout()
fig2.savefig(RESULTS_PATH / "cancer_brier.png", dpi=600)
plt.show()